In [1]:
# import open_clip

# model, preprocess = open_clip.create_model_from_pretrained('hf-hub:laion/CLIP-ViT-g-14-laion2B-s12B-b42K')
# tokenizer = open_clip.get_tokenizer('hf-hub:laion/CLIP-ViT-g-14-laion2B-s12B-b42K')

In [1]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch
from qdrant_client import QdrantClient, models
import uuid

E0000 00:00:1773950977.807536 3777022 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773950977.807549 3777022 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1773950977.807551 3777022 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773950977.807552 3777022 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1773950977.807553 3777022 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
# Универсальный выбор устройства
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

model = AutoModel.from_pretrained(
    "jinaai/jina-clip-v2",
    trust_remote_code=True
)

model = model.to(device)

processor = AutoProcessor.from_pretrained(
    "jinaai/jina-clip-v2",
    trust_remote_code=True
)

Using device: mps


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
/Users/ast/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_clip_hyphen_implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:137: UserWarning: Flash attention requires CUDA, disabling
  warnings.warn('Flash attention requires CUDA, disabling')
/Users/ast/.cache/huggingface/modules/transformers_modules/jinaai/jina_hyphen_clip_hyphen_implementation/39e6a55ae971b59bea6e44675d237c99762e7ee2/modeling_clip.py:172: UserWarning: xFormers requires CUDA, disabling
  warnings.warn('xFormers requires CUDA, disabling')


preprocessor_config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

processing_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- processing_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [3]:
EMBEDDING_SIZE = 1024

In [17]:
def search_by_text(query_text, client, collection_name, model, processor, device, top_k=5):
    inputs = processor(
        text=[query_text],
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    text_embedding = text_features[0].cpu().numpy()

    results = client.query_points(
        collection_name=collection_name,
        query=text_embedding,
        limit=top_k
    )

    return results

In [5]:
def insert_image(img, client, collection_name):
    # Предобработка изображения через процессор Jina
    image_inputs = processor(images=img, return_tensors='pt').to(device)
    
    with torch.no_grad():
        image_features = model.get_image_features(**image_inputs)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    
    image_embedding = image_features.cpu().float().numpy()[0].tolist()
    
    point_id = str(uuid.uuid4())
    client.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=point_id,
                vector=image_embedding,
                payload={
                    "image_path": img.filename,
                    "model": "Jina-CLIP-v2",
                    "dimensions": EMBEDDING_SIZE
                }
            )
        ]
    )

In [6]:
# path = "patrick.jpg"
# image = Image.open(path)
# image = preprocess(image).unsqueeze(0)
# client = QdrantClient(":memory:")

In [7]:
# with torch.no_grad(), torch.amp.autocast("mps"):
#     image_features = model.encode_image(image)
#     image_features /= image_features.norm(dim=-1, keepdim=True)

# image_features.dtype

In [8]:
client = QdrantClient(":memory:")
collection_name = "image_vectors"

# Создаём коллекцию с той же размерностью (1024) и COSINE distance
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_SIZE,
        distance=models.Distance.COSINE
    )
)

True

In [9]:
# image_embedding = image_features.cpu().numpy()[0].tolist()

In [11]:
import os

directory = "./data/"
for f in os.listdir(directory):
    if f.endswith(".jpg"):
        filepath = os.path.join(directory, f)
        image = Image.open(filepath)
        insert_image(image, client, collection_name)

In [33]:
results = search_by_text(
    "не чета не хочу пока",
    client,
    collection_name,
    model,
    processor,
    device,
    top_k=3
)

[(r.payload, r.score) for r in results.points]

[({'image_path': './data/nehochu.jpg',
   'model': 'Jina-CLIP-v2',
   'dimensions': 1024},
  0.4099199662613042),
 ({'image_path': './data/kislinka.jpg',
   'model': 'Jina-CLIP-v2',
   'dimensions': 1024},
  0.3407127846097074),
 ({'image_path': './data/surgut.jpg',
   'model': 'Jina-CLIP-v2',
   'dimensions': 1024},
  0.2961875691437003)]

In [12]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B")
# model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B")

In [ ]:
model.